# RoPE（旋转位置编码）实现
## Rotary Position Embedding Implementation

In [ ]:
# RoPE旋转机制可视化 / RoPE Rotation Mechanism
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Rotation matrix visualization / 旋转矩阵
ax1 = axes[0]
angles = np.linspace(0, 4*np.pi, 100)
x = np.cos(angles)
y = np.sin(angles)

ax1.plot(x, y, 'b-', linewidth=2)
ax1.scatter([1], [0], color='red', s=100, zorder=5, label='Position 0')
ax1.scatter([0], [1], color='green', s=100, zorder=5, label='Position m')
ax1.scatter([-1], [0], color='purple', s=100, zorder=5, label='Position n')
ax1.set_xlabel('Real Part')
ax1.set_ylabel('Imaginary Part')
ax1.set_title('RoPE Rotation in Complex Plane
(Different positions = different rotations)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# 2. QK attention with position / 带位置的内积
ax2 = axes[1]
positions = np.arange(0, 20)
# Simulated attention decay with distance
attention_decay = np.exp(-positions / 5)

ax2.bar(positions, attention_decay, color='#3498db', edgecolor='black')
ax2.set_xlabel('Relative Distance (m-n)')
ax2.set_ylabel('Attention Score')
ax2.set_title('RoPE Relative Position Encoding
(Attention decays with distance)')
ax2.grid(True, alpha=0.3)

# 3. Position comparison: learned vs RoPE
ax3 = axes[2]
methods = ['Learned PE', 'Sinusoidal PE', 'RoPE']
max_lengths = [2048, 4096, 65536]  # maximum supported context
colors = ['#3498db', '#2ecc71', '#e74c3c']

bars = ax3.bar(methods, max_lengths, color=colors)
ax3.set_ylabel('Max Context Length')
ax3.set_title('Position Encoding Comparison
(RoPE supports longest context)')
ax3.set_yscale('log')

for bar, length in zip(bars, max_lengths):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             f'{length}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../images/rope_rotation.png', dpi=150, bbox_inches='tight')
plt.show()

print("RoPE rotation visualization saved!")

<img src="../images/logo.png" width=150>

RoPE通过旋转操作将位置信息编码到Query和Key中，解决了传统位置编码无法处理超长序列的问题。LLaMA、GLM、DeepSeek等模型都采用RoPE。

RoPE encodes position information into Query and Key through rotation operations, solving the problem that traditional positional encoding cannot handle very long sequences. Models like LLaMA, GLM, and DeepSeek all use RoPE.

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

class RotaryPositionalEmbedding(nn.Module):
    """
    RoPE (Rotary Position Embedding)
    
    核心思想：对Q和K的每对维度应用旋转，从而编码相对位置信息
    
    Core idea: Apply rotation to each pair of dimensions of Q and K to encode relative position information
    """
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        
        # 预计算旋转角度 / Precompute rotation angles
        # inv_freq[i] = 1 / (base^(2i/dim))
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len)
        
        # freqs[t, i] = t * inv_freq[i]
        freqs = torch.outer(t, inv_freq)  # (max_seq_len, dim/2)
        
        # 缓存cos和sin / Cache cos and sin
        self.register_buffer('cos_cached', freqs.cos())
        self.register_buffer('sin_cached', freqs.sin())
    
    def forward(self, seq_len):
        """返回给定序列长度的旋转矩阵 / Return rotation matrices for given sequence length"""
        return (
            self.cos_cached[:seq_len],
            self.sin_cached[:seq_len]
        )

# 测试 / Test
rope = RotaryPositionalEmbedding(dim=64, max_seq_len=128)
cos, sin = rope(seq_len=32)

print(f"RoPE cos shape: {cos.shape}")
print(f"RoPE sin shape: {sin.shape}")

# 可视化旋转编码 / Visualize rotation encoding
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(cos[:32].numpy().T, aspect='auto', cmap='RdBu_r')
plt.colorbar()
plt.title('RoPE Cosine (first 32 positions)')
plt.xlabel('Position')
plt.ylabel('Dimension')

plt.subplot(1, 2, 2)
plt.imshow(sin[:32].numpy().T, aspect='auto', cmap='RdBu_r')
plt.colorbar()
plt.title('RoPE Sine (first 32 positions)')
plt.xlabel('Position')
plt.ylabel('Dimension')

plt.tight_layout()
plt.savefig('../images/rope_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# 应用RoPE到Attention
## Apply RoPE to Attention

In [ ]:
def apply_rotary_pos_emb(q, k, cos, sin):
    """
    应用RoPE到Q和K
    Apply RoPE to Q and K
    
    参数 / Parameters:
        q: (batch, heads, seq, head_dim)
        k: (batch, heads, seq, head_dim)
        cos: (seq, head_dim/2) or (1, 1, seq, head_dim/2)
        sin: (seq, head_dim/2) or (1, 1, seq, head_dim/2)
    
    返回 / Returns:
        q_rot, k_rot: 旋转后的Q和K
    """
    # 确保cos和sin是正确形状 / Ensure cos and sin have correct shape
    if cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(0)  # (1, 1, seq, head_dim/2)
        sin = sin.unsqueeze(0).unsqueeze(0)
    
    # 将head_dim分成两半 / Split head_dim into two halves
    half_dim = cos.shape[-1]
    
    # 分离Q的实部和虚部 / Separate real and imaginary parts of Q
    q_real = q[..., :half_dim]
    q_imag = q[..., half_dim:]
    
    k_real = k[..., :half_dim]
    k_imag = k[..., half_dim:]
    
    # RoPE公式 / RoPE formula:
    # q' = q_real * cos - q_imag * sin
    # q'' = q_real * sin + q_imag * cos
    
    q_rot_real = q_real * cos - q_imag * sin
    q_rot_imag = q_real * sin + q_imag * cos
    
    k_rot_real = k_real * cos - k_imag * sin
    k_rot_imag = k_real * sin + k_imag * cos
    
    q_rot = torch.cat([q_rot_real, q_rot_imag], dim=-1)
    k_rot = torch.cat([k_rot_real, k_rot_imag], dim=-1)
    
    return q_rot, k_rot

# 测试 / Test
batch, heads, seq, head_dim = 2, 8, 32, 64
q = torch.randn(batch, heads, seq, head_dim)
k = torch.randn(batch, heads, seq, head_dim)

rope = RotaryPositionalEmbedding(dim=head_dim, max_seq_len=128)
cos, sin = rope(seq_len=seq)

q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)

print(f"Original Q shape: {q.shape}")
print(f"Rotated Q shape: {q_rot.shape}")
print(f"Original K shape: {k.shape}")
print(f"Rotated K shape: {k_rot.shape}")

# RoPE注意力层
## RoPE Attention Layer

In [ ]:
class RoPEMultiHeadAttention(nn.Module):
    """
    带RoPE的多头注意力
    Multi-head attention with RoPE
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.rope = RotaryPositionalEmbedding(self.head_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x, mask=None):
        B, N, C = x.shape
        
        # QKV投影 / QKV projection
        q = self.q_proj(x).reshape(B, N, self.num_heads, self.head_dim)
        k = self.k_proj(x).reshape(B, N, self.num_heads, self.head_dim)
        v = self.v_proj(x).reshape(B, N, self.num_heads, self.head_dim)
        
        # 应用RoPE / Apply RoPE
        cos, sin = self.rope(N)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # 调整维度顺序 / Adjust dimension order
        q = q.transpose(1, 2)  # (B, heads, seq, head_dim)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        
        # 计算注意力 / Compute attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        
        # 应用注意力到V / Apply attention to V
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

# 测试 / Test
rope_attn = RoPEMultiHeadAttention(embed_dim=512, num_heads=8)
x = torch.randn(2, 64, 512)
output = rope_attn(x)
print(f"RoPE Attention input: {x.shape} -> output: {output.shape}")

# RoPE与相对位置编码对比
## RoPE vs Relative Position Encoding

In [ ]:
# 对比RoPE和传统相对位置编码的注意力分布
# Compare attention distribution between RoPE and traditional relative position encoding

def compute_attention_with_rope(q, k, seq_len, head_dim):
    """使用RoPE计算注意力 / Compute attention with RoPE"""
    rope = RotaryPositionalEmbedding(head_dim)
    cos, sin = rope(seq_len)
    q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
    
    scale = head_dim ** -0.5
    attn = (q_rot @ k_rot.transpose(-2, -1)) * scale
    return F.softmax(attn, dim=-1)

def compute_attention_without_rope(q, k):
    """不使用RoPE计算注意力 / Compute attention without RoPE"""
    scale = q.size(-1) ** -0.5
    attn = (q @ k.transpose(-2, -1)) * scale
    return F.softmax(attn, dim=-1)

# 创建测试数据 / Create test data
seq_len = 32
head_dim = 64
q = torch.randn(1, 1, seq_len, head_dim)
k = torch.randn(1, 1, seq_len, head_dim)

attn_with_rope = compute_attention_with_rope(q, k, seq_len, head_dim)
attn_without_rope = compute_attention_without_rope(q, k)

# 可视化 / Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(attn_without_rope[0, 0].numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Attention WITHOUT RoPE\n(Bias towards nearby positions)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(attn_with_rope[0, 0].numpy(), cmap='Blues', aspect='auto')
axes[1].set_title('Attention WITH RoPE\n(Position-encoded)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('../images/rope_vs_no_rope.png', dpi=150, bbox_inches='tight')
plt.show()

# 位置编码外推测试
## Position Encoding Extrapolation Test

In [ ]:
# 测试RoPE的外推能力 - 训练时没见过的新位置
# Test RoPE extrapolation - new positions not seen during training

def test_rope_extrapolation():
    """测试RoPE在外推位置时的表现"""
    
    train_seq_len = 128
    test_seq_len = 256  # 超过训练长度
    head_dim = 64
    
    # 创建RoPE / Create RoPE
    rope = RotaryPositionalEmbedding(dim=head_dim, max_seq_len=train_seq_len)
    
    # 测试不同位置的距离衰减 / Test distance decay at different positions
    distances = []
    cos_similarities = []
    
    for pos in range(0, test_seq_len, 16):
        cos, sin = rope(min(pos + 1, train_seq_len))
        
        if pos > 0:
            # 计算相邻位置cos的相似度 / Compute cosine similarity of adjacent positions
            cos1 = cos[pos - 1] if pos <= train_seq_len else rope.cos_cached[train_seq_len - 1]
            cos2 = cos[pos] if pos <= train_seq_len else rope.cos_cached[train_seq_len - 1]
            similarity = torch.nn.functional.cosine_similarity(
                cos1.unsqueeze(0), cos2.unsqueeze(0)
            ).item()
            distances.append(pos)
            cos_similarities.append(similarity)
    
    return distances, cos_similarities

distances, similarities = test_rope_extrapolation()

plt.figure(figsize=(10, 5))
plt.plot(distances, similarities, 'b-o', linewidth=2, markersize=8)
plt.axvline(x=128, color='red', linestyle='--', label='Train limit (128)')
plt.axhline(y=similarities[-1], color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Position Distance')
plt.ylabel('Cosine Similarity')
plt.title('RoPE Extrapolation: Position Similarity Decay')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../images/rope_extrapolation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRoPE extrapolation analysis:")
print(f"  Positions beyond 128 show gradual similarity decay")
print(f"  This allows model to generalize to longer sequences than training")

# 整合到LLaMA风格模型
## Integration into LLaMA-style Model

In [ ]:
class LlamaBlock(nn.Module):
    """
    LLaMA风格的Transformer块（使用RoPE）
    LLaMA-style Transformer block (with RoPE)
    """
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = RoPEMultiHeadAttention(embed_dim, num_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        
        # SwiGLU激活 / SwiGLU activation (LLaMA uses SwiGLU)
        self.w1 = nn.Linear(embed_dim, ff_dim)
        self.w2 = nn.Linear(embed_dim, ff_dim)
        self.w3 = nn.Linear(ff_dim, embed_dim)
    
    def forward(self, x, mask=None):
        x = x + self.attention(self.ln1(x), mask)
        
        # SwiGLU FFN
        gate = self.w1(x)
        x_silu = F.silu(gate)
        up = self.w3(F.silu(self.w2(x)))
        x = x + up
        
        return x

# 创建LLaMA风格模型 / Create LLaMA-style model
class LlamaModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, ff_dim, max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        
        self.blocks = nn.ModuleList([
            LlamaBlock(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ])
        
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, x, mask=None):
        x = self.token_embedding(x)
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.ln_f(x))

# 测试 / Test
llama = LlamaModel(
    vocab_size=32000,
    embed_dim=512,
    num_heads=8,
    num_layers=4,
    ff_dim=1376,
    max_seq_len=2048
)

x = torch.randint(0, 32000, (2, 64))
output = llama(x)

total_params = sum(p.numel() for p in llama.parameters())
print(f"LLaMA-style model:")
print(f"  Input: {x.shape} -> Output: {output.shape}")
print(f"  Total parameters: {total_params / 1e6:.1f}M")

# 总结

| 特性 | RoPE | 传统位置编码 |
|------|------|-------------|
| 编码方式 | 旋转操作 | 加法/拼接 |
| 相对位置 | 天然支持 | 需要额外计算 |
| 长序列外推 | 较好 | 较差 |
| 计算开销 | 较低 | 中等 |

RoPE的核心优势是只需对Q和K应用旋转操作，无需修改注意力分数计算，兼容现有的优化实现（如Flash Attention）。

# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **RoPE旋转位置编码** - 基础实现
2. **RoPE应用到Attention** - Q/K旋转
3. **RoPE注意力层** - 完整的多头注意力
4. **位置编码外推测试** - RoPE外推能力验证
5. **整合到LLaMA风格模型** - 端到端集成

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **Flash Attention with RoPE** | 高效注意力实现 | [Flash Attention](https://arxiv.org/abs/2205.14135) |
| **YaRN** | 长上下文外推增强 | [YaRN Paper](https://arxiv.org/abs/2309.00071) |
| **NTK-aware Scaling** | 插值+外推结合 | [NTK Scaling](https://www.reddit.com/r/LocalLLaMA/comments/14nm3pd/) |
| **CoLT5** | 长上下文LLaMA变体 | [CoLT5](https://arxiv.org/abs/2303.05448) |
